---

**Setting in PGL**

---

In [ ]:
%load_ext autoreload
%autoreload 2
from pgl import pgl
pgl = pgl()
pgl.cleanUp()

---

**Display settings**

---
Settings for individual displays

In [ ]:
pgl.displaySettings()

---

**Overall settings for PGL**

---

You use these to create different settings that are used by experiments.


In [ ]:
pgl.settings()

---

Eye tracker settings

---

In [ ]:
pgl.eyeTrackerSettings()

---

Stencil settings

---

In [ ]:
pgl.stencilSettings()

In [4]:
def testLabJackShortPulses(duration=5.0):
    """
    Capture FIO0 at a requested 100 kHz and report both edges and pulse widths.

    Uses the verified FIO_STATE stream register. Opens a dedicated T7 USB
    connection; close other LabJack connections before running.

    Times are relative to acquisition sample zero, not calibrated host times.
    HIGH/LOW widths include only intervals bounded by two acquired edges.

    Returns:
        dict: Samples, edge records, pulse widths, and stream diagnostics.
    """
    import time
    import numpy as np
    from labjack import ljm
    from pgl.pglLabJack import pglLabJack

    if isinstance(duration, (bool, np.bool_)):
        raise ValueError("duration must be finite and positive")
    duration = float(duration)
    if not np.isfinite(duration) or duration <= 0:
        raise ValueError("duration must be finite and positive")

    # Request the T7's advertised maximum stream sampling rate.
    # One bank register per scan; no analog channels or other banks.
    requestedScanRate = 100000.0
    scansPerRead = 1000
    handle = None
    streamStarted = False
    batches = []
    readDurations = []
    readCompletionTimes = []
    backlogs = []
    cleanupErrors = []
    acquiredSamples = 0

    try:
        handle = ljm.openS("T7", "USB", "ANY")
        print(f"Device: {ljm.getHandleInfo(handle)}")

        # Reading FIO0 configures only that pin as an input.
        # Other FIO pins are sampled but their directions are not changed here.
        ljm.eReadName(handle, "FIO0")

        # Use internal clocking and immediate stream start.
        ljm.eWriteName(handle, "STREAM_TRIGGER_INDEX", 0)
        ljm.eWriteName(handle, "STREAM_CLOCK_SOURCE", 0)
        streamAddress, streamType = ljm.nameToAddress("FIO_STATE")
        print(f"FIO_STATE: address={streamAddress}, dataType={streamType}")

        print(f"Starting FIO0 capture: requested {requestedScanRate:g} scans/s.")
        startBefore = time.perf_counter()
        actualScanRate = ljm.eStreamStart(handle, scansPerRead, 1, [streamAddress], requestedScanRate)
        streamStarted = True
        startAfter = time.perf_counter()

        if not np.isfinite(actualScanRate) or actualScanRate <= 0:
            raise RuntimeError(f"LJM returned an invalid scan rate: {actualScanRate}")

        print(f"Actual rate: {actualScanRate:g} scans/s; sample spacing: {1e6 / actualScanRate:.3f} µs.")
        if not np.isclose(actualScanRate, requestedScanRate, rtol=1e-6, atol=0):
            raise RuntimeError("The device did not provide the requested 100 kHz rate; stopping rather than silently testing slower.")

        targetSamples = int(np.ceil(duration * actualScanRate))

        # The deadline limits the loop, but a blocking native read still
        # depends on LJM's own timeout behavior.
        deadline = time.monotonic() + duration + 10.0

        while acquiredSamples < targetSamples:
            if time.monotonic() > deadline:
                raise TimeoutError("Digital capture exceeded its acquisition deadline")

            readBefore = time.perf_counter()
            values, deviceBacklog, ljmBacklog = ljm.eStreamRead(handle)
            readAfter = time.perf_counter()

            batch = np.asarray(values, dtype=np.float64)
            if batch.ndim != 1 or batch.size == 0:
                raise RuntimeError("LJM returned an empty or malformed stream batch")

            # Never drop missing-data markers: doing so would shift timestamps
            # and potentially invent edges across an acquisition gap.
            skippedRows = np.flatnonzero(batch == -9999)
            if skippedRows.size:
                sampleIndex = acquiredSamples + int(skippedRows[0])
                raise RuntimeError(
                    f"LJM skipped-sample marker at index {sampleIndex}; "
                    f"{skippedRows.size} markers in this batch. "
                    f"Device / LJM backlog: {deviceBacklog} / {ljmBacklog}. "
                    "Capture is incomplete."
                )

            # Validate the full UINT16 word before extracting FIO0.
            # Do not reject 0xFFFF merely because it exceeds an 8-bit bank value.
            invalid = ~np.isfinite(batch) | (batch < 0) | (batch > 65535) | (batch != np.floor(batch))
            if np.any(invalid):
                badRow = int(np.flatnonzero(invalid)[0])
                raise RuntimeError(
                    f"Invalid digital stream word at index {acquiredSamples + badRow}: "
                    f"{batch[badRow]}. Capture is incomplete."
                )

            # Preserve the raw words so unexpected upper bits remain inspectable.
            batches.append(batch.astype(np.uint16))
            acquiredSamples += batch.size
            readDurations.append(readAfter - readBefore)
            readCompletionTimes.append(readAfter)
            backlogs.append((int(deviceBacklog), int(ljmBacklog)))

    finally:
        # Attempt both cleanup operations without masking an acquisition error.
        if handle is not None:
            if streamStarted:
                try:
                    ljm.eStreamStop(handle)
                except Exception as error:
                    cleanupErrors.append(f"Stream stop failed: {type(error).__name__}: {error}")

            try:
                ljm.close(handle)
            except Exception as error:
                cleanupErrors.append(f"Connection close failed: {type(error).__name__}: {error}")

        for message in cleanupErrors: print(message)

    if cleanupErrors: raise RuntimeError("; ".join(cleanupErrors))

    # Retain the final complete batch, which may extend slightly past duration.
    states = np.concatenate(batches)

    # Decode only the connected input. Upper bits must not create FIO0 edges.
    fio0States = (states & 1).astype(np.uint8)
    events, _ = pglLabJack._digitalEventsFromSamples(
        fio0States, {"FIO0": 0}, None, 0, actualScanRate, 0.0, edge="both"
    )

    # Return plain records: the zero time anchor above is relative, not a
    # production pglTimestamp host-clock mapping.
    edges = [
        {"edge": event.eventType, "sampleIndex": event.sampleIndex, "sampleTime": event.sampleTime}
        for event in events
    ]
    risingIndices = np.array([event.sampleIndex for event in events if event.eventType == "rising"], dtype=np.int64)
    fallingIndices = np.array([event.sampleIndex for event in events if event.eventType == "falling"], dtype=np.int64)

    # Consecutive opposite edges bound complete HIGH or LOW intervals.
    # Exclude partial intervals at the beginning and end of acquisition.
    highWidths = []
    lowWidths = []
    for first, second in zip(events, events[1:]):
        width = (second.sampleIndex - first.sampleIndex) / actualScanRate
        if first.eventType == "rising" and second.eventType == "falling":
            highWidths.append(width)
        elif first.eventType == "falling" and second.eventType == "rising":
            lowWidths.append(width)

    highWidths = np.asarray(highWidths)
    lowWidths = np.asarray(lowWidths)
    readDurations = np.asarray(readDurations)
    readCompletionTimes = np.asarray(readCompletionTimes)
    backlogs = np.asarray(backlogs)
    highFraction = float(np.mean(fio0States))

    print("\nCapture complete")
    print(f"Samples: {states.size}; sampled duration: {states.size / actualScanRate:.6f} s")
    print(f"FIO0 first / last state: {int(fio0States[0])} / {int(fio0States[-1])}")
    print(f"Fraction of samples HIGH: {100 * highFraction:.6f}%")
    print(f"Rising edges: {risingIndices.size}; falling edges: {fallingIndices.size}")
    print(f"Stream-start call duration: {(startAfter - startBefore) * 1000:.3f} ms")
    print(f"Median / maximum read duration: {np.median(readDurations) * 1000:.3f} / {np.max(readDurations) * 1000:.3f} ms")
    print(f"Maximum device / LJM backlog: {backlogs[:, 0].max()} / {backlogs[:, 1].max()} scans")

    # Inspect unexpected upper bits rather than silently discarding evidence.
    upperBitCount = int(np.count_nonzero(states > 255))
    print(f"Raw words with upper eight bits set: {upperBitCount}")
    if upperBitCount:
        firstUpperRow = int(np.flatnonzero(states > 255)[0])
        left = max(0, firstUpperRow - 5)
        right = min(states.size, firstUpperRow + 6)
        print(f"First upper-bit occurrence: sample {firstUpperRow}")
        print(f"Raw words at samples {left}–{right - 1}: {[f'0x{int(value):04X}' for value in states[left:right]]}")

    highValues, highCounts = np.unique(states[fio0States != 0], return_counts=True)
    print(f"Raw words while FIO0 HIGH (first 20 distinct values): {[(f'0x{int(value):04X}', int(count)) for value, count in zip(highValues[:20], highCounts[:20])]}")

    for label, widths in (("HIGH", highWidths), ("LOW", lowWidths)):
        if widths.size:
            widthsUs = widths * 1e6
            print(f"{label} intervals: {widths.size}; min / median / max: {widthsUs.min():.3f} / {np.median(widthsUs):.3f} / {widthsUs.max():.3f} µs")
            print(f"  First widths (µs): {widthsUs[:20].tolist()}")
        else:
            print(f"{label} intervals: no complete intervals captured")

    # Repeated same-polarity edges measure the pulse repetition interval.
    for label, indices in (("Rising", risingIndices), ("Falling", fallingIndices)):
        if indices.size > 1:
            intervals = np.diff(indices) / actualScanRate
            print(f"{label}-edge spacing, min / median / max: {intervals.min():.6f} / {np.median(intervals):.6f} / {intervals.max():.6f} s")

    print("First edges (seconds relative to sample zero):")
    for edge in edges[:20]:
        print(f"  {edge['edge']:7s} sample={edge['sampleIndex']:8d} time={edge['sampleTime']:.6f}")

    if not events:
        print("No transitions detected. This does not prove no pulses occurred: check wiring, signal timing, levels, and pulse width.")

    return {
        "scanRate": actualScanRate,
        "states": states,
        "fio0States": fio0States,
        "edges": edges,
        "highWidths": highWidths,
        "lowWidths": lowWidths,
        "readDurations": readDurations,
        "readCompletionTimes": readCompletionTimes,
        "backlogs": backlogs,
        "startCallDuration": startAfter - startBefore,
    }

In [5]:
pulseTest = testLabJackShortPulses(duration=60.0)

Device: (7, 1, 470040275, 0, 0, 64)
FIO_STATE: address=2500, dataType=0
Starting FIO0 capture: requested 100000 scans/s.
Actual rate: 100000 scans/s; sample spacing: 10.000 µs.

Capture complete
Samples: 6000000; sampled duration: 60.000000 s
FIO0 first / last state: 0 / 0
Fraction of samples HIGH: 0.001033%
Rising edges: 62; falling edges: 62
Stream-start call duration: 10.145 ms
Median / maximum read duration: 9.878 / 61.085 ms
Maximum device / LJM backlog: 963 / 672 scans
Raw words with upper eight bits set: 2
First upper-bit occurrence: sample 23934
Raw words at samples 23929–23939: ['0x00FE', '0x00FE', '0x00FE', '0x00FE', '0x00FE', '0xFFFF', '0x00FE', '0x00FE', '0x00FE', '0x00FE', '0x00FE']
Raw words while FIO0 HIGH (first 20 distinct values): [('0x00FF', 60), ('0xFFFF', 2)]
HIGH intervals: 62; min / median / max: 10.000 / 10.000 / 10.000 µs
  First widths (µs): [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]

In [6]:
def inspectLabJackPulseWords(pulseTest):
    """Inspect expected FIO0 transitions and anomalous words without altering data."""
    import numpy as np

    states = np.asarray(pulseTest["states"])
    scanRate = float(pulseTest["scanRate"])

    # This recording's observed baseline is 0x00FE. Its ordinary HIGH
    # samples are 0x00FF; this classification is diagnostic, not a general
    # replacement for the production edge detector.
    ordinaryHighIndices = np.flatnonzero(states == 0x00FF)
    unexpectedIndices = np.flatnonzero(states > 0x00FF)

    # Identify starts of ordinary HIGH runs without compressing sample time.
    ordinaryStarts = ordinaryHighIndices[
        np.concatenate(([True], np.diff(ordinaryHighIndices) > 1))
    ] if ordinaryHighIndices.size else np.array([], dtype=np.int64)

    print(f"Ordinary HIGH samples: {ordinaryHighIndices.size}")
    print(f"Ordinary HIGH runs: {ordinaryStarts.size}")
    print(f"Unexpected upper-bit samples: {unexpectedIndices.size}")

    if ordinaryStarts.size > 1:
        intervals = np.diff(ordinaryStarts) / scanRate
        print(f"Ordinary pulse spacing, min / median / max: {intervals.min():.6f} / {np.median(intervals):.6f} / {intervals.max():.6f} s")

    # Preserve the original acquisition indices when inspecting anomalies.
    for index in unexpectedIndices[:20]:
        index = int(index)
        left = max(0, index - 3)
        right = min(states.size, index + 4)
        nearbyWords = [f"0x{int(value):04X}" for value in states[left:right]]
        print(f"\nUnexpected word at sample {index}, time {index / scanRate:.6f} s")
        print(f"  Samples {left}–{right - 1}: {nearbyWords}")

        if ordinaryStarts.size:
            nearest = int(ordinaryStarts[np.argmin(np.abs(ordinaryStarts - index))])
            print(f"  Offset from nearest ordinary pulse: {(index - nearest) / scanRate:.6f} s")

    return {"ordinaryPulseIndices": ordinaryStarts, "unexpectedIndices": unexpectedIndices}

In [7]:
wordCheck = inspectLabJackPulseWords(pulseTest)

Ordinary HIGH samples: 60
Ordinary HIGH runs: 60
Unexpected upper-bit samples: 2
Ordinary pulse spacing, min / median / max: 0.932790 / 1.000000 / 1.000000 s

Unexpected word at sample 23934, time 0.239340 s
  Samples 23931–23937: ['0x00FE', '0x00FE', '0x00FE', '0xFFFF', '0x00FE', '0x00FE', '0x00FE']
  Offset from nearest ordinary pulse: -0.385060 s

Unexpected word at sample 5382702, time 53.827020 s
  Samples 5382699–5382705: ['0x00FE', '0x00FE', '0x00FE', '0xFFFF', '0x00FE', '0x00FE', '0x00FE']
  Offset from nearest ordinary pulse: 0.202720 s


In [8]:
def inspectLabJackPulseTiming(pulseTest, expectedPeriod=1.0, tolerance=0.0001):
    """
    Locate timing anomalies without modifying or filtering the acquisition.

    expectedPeriod is a diagnostic assumption about the pulse source.
    This test cannot distinguish source jitter from acquisition problems.
    """
    import numpy as np

    states = np.asarray(pulseTest["states"])
    scanRate = float(pulseTest["scanRate"])

    # Locate ordinary HIGH runs using original sample indices.
    ordinaryHigh = states == 0x00FF
    ordinaryStarts = np.flatnonzero(ordinaryHigh & np.concatenate(([True], ~ordinaryHigh[:-1])))
    intervals = np.diff(ordinaryStarts) / scanRate
    unusualIntervals = np.flatnonzero(np.abs(intervals - expectedPeriod) > tolerance)

    print(f"Ordinary pulse starts: {ordinaryStarts.size}")
    print(f"Intervals outside {expectedPeriod:g} ± {tolerance:g} seconds: {unusualIntervals.size}")

    for intervalIndex in unusualIntervals[:20]:
        first = int(ordinaryStarts[intervalIndex])
        second = int(ordinaryStarts[intervalIndex + 1])
        print(
            f"  Pulses {intervalIndex + 1} → {intervalIndex + 2}: "
            f"samples {first} → {second}; "
            f"times {first / scanRate:.6f} → {second / scanRate:.6f} s; "
            f"interval={intervals[intervalIndex]:.6f} s"
        )

    # Reconstruct all bit-zero rising edges, including anomalous raw words.
    # Compare against the events saved by the original capture function.
    fio0High = (states & 1) != 0
    rawRisingIndices = np.flatnonzero(fio0High[1:] & ~fio0High[:-1]) + 1
    savedRisingIndices = np.array(
        [edge["sampleIndex"] for edge in pulseTest["edges"] if edge["edge"] == "rising"],
        dtype=np.int64
    )
    recordsMatch = np.array_equal(rawRisingIndices, savedRisingIndices)
    print(f"Saved rising-edge indices match raw samples: {recordsMatch}")

    # Show overall counts without deleting suspect samples.
    print(f"Raw bit-zero rising edges: {rawRisingIndices.size}")
    print(f"Unexpected upper-bit samples: {np.count_nonzero(states > 255)}")

    return {"ordinaryStarts": ordinaryStarts, "intervals": intervals, "unusualIntervals": unusualIntervals, "recordsMatch": recordsMatch}

In [9]:
timingCheck = inspectLabJackPulseTiming(pulseTest)

Ordinary pulse starts: 60
Intervals outside 1 ± 0.0001 seconds: 1
  Pulses 54 → 55: samples 5362430 → 5455709; times 53.624300 → 54.557090 s; interval=0.932790 s
Saved rising-edge indices match raw samples: True
Raw bit-zero rising edges: 62
Unexpected upper-bit samples: 2


In [ ]:
def testLabJackThreadedTriggers(duration=60.0, pollInterval=0.005):
    """
    Test threaded scanner-trigger capture on FIO0 without an experiment.

    Requests 100 kHz sampling with 100 scans/read: nominally 1 ms per batch.
    The calling thread polls the event queue independently of acquisition.

    Args:
        duration (float): Requested sampled duration in seconds.
        pollInterval (float): Approximate delay between queue polls, in seconds.

    Returns:
        dict: Trigger records, raw samples, uncertain sample indices,
            timing diagnostics, and acquisition status.

    Opens one dedicated T7 USB connection. Close other LabJack connections first.

    Unknown words preserve their sample positions but reset edge comparison.
    A real edge coinciding with an unknown sample may therefore be missed.

    Sample times are acquisition-relative. Estimated host timestamps use the
    midpoint of the stream-start call and are NOT calibrated edge timestamps.
    Read completion and polling timestamps describe delivery, not pulse onset.
    """
    import queue
    import threading
    import time
    import numpy as np
    from labjack import ljm
    from pgl.pglLabJack import pglLabJack
    from pgl.pglTimestamp import pglTimestamp

    # Validate before starting a worker or changing hardware.
    for name, value in (("duration", duration), ("pollInterval", pollInterval)):
        if isinstance(value, (bool, np.bool_)):
            raise ValueError(f"{name} must be finite and positive")
        if not np.isfinite(float(value)) or float(value) <= 0:
            raise ValueError(f"{name} must be finite and positive")

    duration = float(duration)
    pollInterval = float(pollInterval)
    requestedScanRate = 100000.0
    scansPerRead = 100

    # Only the worker touches hardware. Only the polling thread collects
    # delivered events. The queue transfers completed event batches.
    eventQueue = queue.Queue(maxsize=10000)
    stopEvent = threading.Event()
    readyEvent = threading.Event()
    finishedEvent = threading.Event()

    # Worker-owned data is inspected only after the worker has finished,
    # except startup information published through readyEvent.
    capture = {
        "scanRate": None,
        "startCallBefore": None,
        "startCallAfter": None,
        "sampleCount": 0,
        "targetSamples": 0,
        "error": None,
        "cleanupErrors": [],
    }
    rawBatches = []
    unknownBatches = []
    readDurations = []
    readCompletionTimes = []
    backlogs = []
    triggers = []

    ################################################################
    # Hardware acquisition and sample-to-event conversion
    ################################################################
    def acquireTriggers():
        handle = None
        streamStarted = False
        previousState = None
        firstSampleIndex = 0

        try:
            handle = ljm.openS("T7", "USB", "ANY")

            # Configure only FIO0 as input; do not alter other pin directions.
            ljm.eReadName(handle, "FIO0")
            ljm.eWriteName(handle, "STREAM_TRIGGER_INDEX", 0)
            ljm.eWriteName(handle, "STREAM_CLOCK_SOURCE", 0)
            streamAddress = ljm.nameToAddress("FIO_STATE")[0]

            # Keep startup timing explicitly separate from sample resolution.
            capture["startCallBefore"] = pglTimestamp.getSecs()
            actualScanRate = ljm.eStreamStart(
                handle, scansPerRead, 1, [streamAddress], requestedScanRate
            )
            streamStarted = True
            capture["startCallAfter"] = pglTimestamp.getSecs()
            capture["scanRate"] = actualScanRate

            if not np.isfinite(actualScanRate) or actualScanRate <= 0:
                raise RuntimeError(f"Invalid scan rate returned by LJM: {actualScanRate}")
            if not np.isclose(actualScanRate, requestedScanRate, rtol=1e-6, atol=0):
                raise RuntimeError(f"Requested 100 kHz but received {actualScanRate:g} Hz")

            capture["targetSamples"] = int(np.ceil(duration * actualScanRate))
            estimatedStartTimestamp = (capture["startCallBefore"] + capture["startCallAfter"]) / 2
            readyEvent.set()

            # A native blocking read still relies on LJM's timeout behavior.
            deadline = time.monotonic() + duration + 15.0

            while firstSampleIndex < capture["targetSamples"] and not stopEvent.is_set():
                if time.monotonic() > deadline:
                    raise TimeoutError("Acquisition exceeded its deadline")

                readBefore = time.perf_counter()
                values, deviceBacklog, ljmBacklog = ljm.eStreamRead(handle)
                readAfter = time.perf_counter()
                readCompletedTimestamp = pglTimestamp.getSecs()

                batch = np.asarray(values, dtype=np.float64)
                if batch.ndim != 1 or batch.size == 0:
                    raise RuntimeError("LJM returned an empty or malformed batch")

                readDurations.append(readAfter - readBefore)
                readCompletionTimes.append(readCompletedTimestamp)
                backlogs.append((int(deviceBacklog), int(ljmBacklog)))

                # Preserve the original samples, including diagnostic markers.
                # Do not cast negative missing-data markers to unsigned integers.
                rawBatches.append(batch)

                skippedRows = np.flatnonzero(batch == -9999)
                if skippedRows.size:
                    raise RuntimeError(
                        f"LJM skipped-sample marker at index "
                        f"{firstSampleIndex + int(skippedRows[0])}. "
                        "Acquisition is incomplete."
                    )

                invalid = ~np.isfinite(batch) | (batch < 0) | (batch > 65535) | (batch != np.floor(batch))
                if np.any(invalid):
                    row = int(np.flatnonzero(invalid)[0])
                    raise RuntimeError(
                        f"Invalid sample at index {firstSampleIndex + row}: {batch[row]}"
                    )

                # Unknown upper-bit words are not accepted as scanner triggers.
                # Their positions remain in the sample timeline.
                unknownRows = np.flatnonzero(batch > 255)
                if unknownRows.size: unknownBatches.append(firstSampleIndex + unknownRows)

                # Process uninterrupted valid spans. Reset at each unknown word
                # rather than inventing a transition across the uncertainty.
                batchEvents = []
                spanStart = 0

                for spanEnd in list(unknownRows) + [batch.size]:
                    spanEnd = int(spanEnd)

                    if spanEnd > spanStart:
                        events, previousState = pglLabJack._digitalEventsFromSamples(
                            batch[spanStart:spanEnd],
                            {"FIO0": 0},
                            previousState,
                            firstSampleIndex + spanStart,
                            actualScanRate,
                            0.0,
                            edge="rising",
                        )

                        for event in events:
                            batchEvents.append({
                                "channel": "FIO0",
                                "edge": "rising",
                                "sampleIndex": event.sampleIndex,
                                "sampleTime": event.sampleTime,
                                "estimatedTimestamp": estimatedStartTimestamp + event.sampleTime,
                                "timestampMethod": "streamStartMidpointEstimate",
                                "readCompletedTimestamp": readCompletedTimestamp,
                            })

                    if spanEnd < batch.size:
                        previousState = None
                        spanStart = spanEnd + 1

                # A full queue is a failure, never silent event loss.
                if batchEvents:
                    try:
                        eventQueue.put_nowait(batchEvents)
                    except queue.Full:
                        raise RuntimeError("Trigger queue overflow; acquisition is incomplete") from None

                firstSampleIndex += batch.size
                capture["sampleCount"] = firstSampleIndex

        except Exception as error:
            # Store text rather than retaining a live exception traceback.
            capture["error"] = f"{type(error).__name__}: {error}"

        finally:
            # The worker owns shutdown too. No other thread closes a handle
            # while eStreamRead may still be using it.
            if handle is not None:
                if streamStarted:
                    try:
                        ljm.eStreamStop(handle)
                    except Exception as error:
                        capture["cleanupErrors"].append(f"Stream stop: {type(error).__name__}: {error}")

                try:
                    ljm.close(handle)
                except Exception as error:
                    capture["cleanupErrors"].append(f"Connection close: {type(error).__name__}: {error}")

            readyEvent.set()
            finishedEvent.set()

    ################################################################
    # Simulated experiment polling: no hardware calls
    ################################################################
    def pollTriggers():
        polledTimestamp = pglTimestamp.getSecs()

        while True:
            try:
                batchEvents = eventQueue.get_nowait()
            except queue.Empty:
                break

            for event in batchEvents: event["polledTimestamp"] = polledTimestamp
            triggers.extend(batchEvents)

    worker = threading.Thread(target=acquireTriggers, name="LabJackTriggerTest", daemon=True)
    worker.start()

    try:
        # Startup failures also set readyEvent so they reach the main thread.
        if not readyEvent.wait(timeout=15.0):
            raise TimeoutError("LabJack worker did not finish startup within 15 seconds")

        if capture["scanRate"] is not None:
            print(f"Actual scan rate: {capture['scanRate']:g} Hz")
            print(f"Requested batch size: {scansPerRead} scans")
            print(f"Polling interval: approximately {pollInterval * 1000:g} ms")
            print("Capturing scanner triggers...")

        while not finishedEvent.is_set():
            pollTriggers()
            finishedEvent.wait(timeout=pollInterval)

        worker.join()
        pollTriggers()

    finally:
        stopEvent.set()

        # Never close hardware from here while the worker might be reading.
        worker.join(timeout=5.0)
        if worker.is_alive():
            print(
                "WARNING: acquisition worker has not stopped. "
                "Do not open another LabJack connection; native read/startup "
                "or cleanup may still be blocked."
            )

    if worker.is_alive():
        raise RuntimeError("LabJack worker is still active; capture cannot be summarized safely")

    ################################################################
    # Report capture integrity and delivery separately
    ################################################################
    states = np.concatenate(rawBatches) if rawBatches else np.array([], dtype=np.float64)
    unknownIndices = np.concatenate(unknownBatches) if unknownBatches else np.array([], dtype=np.int64)
    readDurations = np.asarray(readDurations)
    readCompletionTimes = np.asarray(readCompletionTimes)
    backlogs = np.asarray(backlogs, dtype=np.int64).reshape(-1, 2)

    completed = (
        capture["error"] is None
        and not capture["cleanupErrors"]
        and capture["sampleCount"] >= capture["targetSamples"]
        and capture["scanRate"] is not None
    )

    print("\nCapture finished")
    print(f"Acquisition and cleanup completed: {completed}")
    print(f"Accepted rising edges: {len(triggers)}")
    print(f"Unknown samples: {unknownIndices.size}")

    if capture["scanRate"] is not None:
        print(f"Processed sampled duration: {capture['sampleCount'] / capture['scanRate']:.6f} s")

    if len(triggers) > 1:
        intervals = np.diff([event["sampleTime"] for event in triggers])
        print(f"Trigger spacing, min / median / max: {intervals.min():.6f} / {np.median(intervals):.6f} / {intervals.max():.6f} s")

    if triggers:
        pollDelays = np.array([
            event["polledTimestamp"] - event["readCompletedTimestamp"]
            for event in triggers
        ])
        print(f"Read-completion → polling delay, median / max: {np.median(pollDelays) * 1000:.3f} / {pollDelays.max() * 1000:.3f} ms")

    if readDurations.size:
        print(f"Read duration, median / max: {np.median(readDurations) * 1000:.3f} / {readDurations.max()*1000:.3f} ms")
        print(f"Maximum device / LJM backlog: {backlogs[:, 0].max()} / {backlogs[:, 1].max()} scans")

    if unknownIndices.size:
        print(f"First unknown sample indices: {unknownIndices[:20].tolist()}")
        print("WARNING: trigger detection is uncertain at these samples and the following baseline samples.")

    if capture["error"] is not None: print(f"ACQUISITION ERROR: {capture['error']}")
    for error in capture["cleanupErrors"]: print(f"CLEANUP ERROR: {error}")

    print("First accepted trigger times:", [event["sampleTime"] for event in triggers[:10]])
    print("Host edge timestamps are estimates; reported polling delays exclude device/USB acquisition latency.")

    return {
        "completed": completed,
        "scanRate": capture["scanRate"],
        "triggers": triggers,
        "states": states,
        "unknownIndices": unknownIndices,
        "readDurations": readDurations,
        "readCompletionTimes": readCompletionTimes,
        "backlogs": backlogs,
        "startCallBefore": capture["startCallBefore"],
        "startCallAfter": capture["startCallAfter"],
        "error": capture["error"],
        "cleanupErrors": capture["cleanupErrors"],
    }

SyntaxError: f-string: single '}' is not allowed (2995391543.py, line 303)

In [10]:
import numpy as np
np.savez_compressed("labjackFIO0_100kHz_diagnostic.npz", states=pulseTest["states"], scanRate=pulseTest["scanRate"], readDurations=pulseTest["readDurations"], readCompletionTimes=pulseTest["readCompletionTimes"], backlogs=pulseTest["backlogs"])